In [1]:
import pandas as pd 
import numpy as np 
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [3]:
import pyecoacc as acc 
from pyecoacc.models.pipeline import make_classifier_pipeline
from pyecoacc.util.analytics import compare_models_cv
from pyecoacc.features.transform import ACCStatsTransformer
from pyecoacc.models.deep.cnn import make_cnn_model

# Load data 

In [4]:
molerats_data = pd.read_csv("data/molerats.csv", index_col=0)

In [5]:
ids = molerats_data.Animal 
molerats_data.drop("Animal", inplace=True, axis=1)

X = molerats_data.iloc[:, :-1].values 
y = molerats_data.Behavior.values 

# Define models 

### Make a random forest model pipeline 
- ACCStatsTransformer to compute features from the raw ACC signal 
- no scaling 
- select the top 50 features (f-test)

In [6]:
rf_model = make_classifier_pipeline(features=ACCStatsTransformer(), 
                                    model=RandomForestClassifier(n_estimators=250, max_depth=10),
                                    feature_scaler=False,
                                    feature_selector=True, k_selection=50) 

In [7]:
rf_model

Pipeline(steps=[('features', ACCStatsTransformer()),
                ('selection', SelectKBest(k=50)),
                ('model',
                 RandomForestClassifier(max_depth=10, n_estimators=250))])

### Mkae an XGBoost model

In [8]:
xg_model = make_classifier_pipeline(features=ACCStatsTransformer(), 
                                    model=XGBClassifier(n_estimators=250)) 

In [9]:
xg_model

Pipeline(steps=[('features', ACCStatsTransformer()),
                ('model',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=None, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=False, eval_metric=None,
                               feature_types=None, gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=None, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=250, n_jobs=None,
                               num_parallel_tree=None, random_state=None, ...))])

### Make a couple of CNN models
- the pipeline includes rehspaing to 3-chanels (X, Y, Z) before entering the CNN 
- the default CNN has 4 conv layers with [32, 64, 128, 256] filters respectively, followed by 2 layers of sizes [256, 512] fully connected layers.
- a tiny CNN with only [32, 64] conv filters and a single size 20 fully connected layer. This is shown here as an exmaple of customizing CNNs. 

In [ ]:
cnn_model = make_cnn_model(sequence_length=X.shape[1]//3, 
                           num_behav=np.unique(y).shape[0], 
                           verbose=0)

In [11]:
cnn_model

Pipeline(steps=[('reshape', CNNInputReshaper()),
                ('CNN',
                 NeuralNetClassifier(_params_to_validate={'module__fc_layers', 'module__kernel_size', 'module__conv_filters', 'optimizer__weight_decay', 'module__num_classes', 'module__sequence_length'}, batch_size=128, callbacks=[<skorch.callbacks.training.EarlyStopping object at 0x1774fd750>, <skorch.callbacks.training.C...ers=[32, 64, 128, 256], module__fc_layers=[256, 512], module__kernel_size=5, module__num_classes=6, module__sequence_length=50, optimizer=<class 'torch.optim.adam.Adam'>, optimizer__weight_decay=0.0005, predict_nonlinearity='auto', torch_load_kwargs=None, train_split=<skorch.dataset.ValidSplit object at 0x1774cf4d0>, use_caching='auto', verbose=0, warm_start=False))])

In [ ]:
tiny_cnn_model = make_cnn_model(sequence_length=X.shape[1]//3, 
                                num_behav=np.unique(y).shape[0], 
                                conv_filters=[32, 64],
                                fc_layers=[20],
                                verbose=0)

In [13]:
tiny_cnn_model

Pipeline(steps=[('reshape', CNNInputReshaper()),
                ('CNN',
                 NeuralNetClassifier(_params_to_validate={'module__fc_layers', 'module__kernel_size', 'module__conv_filters', 'optimizer__weight_decay', 'module__num_classes', 'module__sequence_length'}, batch_size=128, callbacks=[<skorch.callbacks.training.EarlyStopping object at 0x1774cff50>, <skorch.callbacks.training.C...le__conv_filters=[32, 64], module__fc_layers=[20], module__kernel_size=5, module__num_classes=6, module__sequence_length=50, optimizer=<class 'torch.optim.adam.Adam'>, optimizer__weight_decay=0.0005, predict_nonlinearity='auto', torch_load_kwargs=None, train_split=<skorch.dataset.ValidSplit object at 0x177483ed0>, use_caching='auto', verbose=0, warm_start=False))])

In [14]:
model_dict = {
    "random forest": rf_model,
    "XGBoost": xg_model,
    "CNN": cnn_model,
    "tiny-CNN": tiny_cnn_model
}

# Compare 
- compare_models_cv uses *GroupKFold* and *classification_report* from sklearn for each model in the model_dict and combines results to form accuracy, precision, recall, and f1 tables comparing the models. 

In [15]:
accuracy, recall, precision, f1, _  = compare_models_cv(X, y, model_dict, cv=3, 
                                                        cv_method="animal-groups", individuals=ids)

Starting model random forest...
Starting model XGBoost...
Starting model CNN...
Starting model tiny-CNN...


In [16]:
accuracy

,random forest,XGBoost,CNN,tiny-CNN
split-1,0.804624,0.824855,0.728902,0.710983
split-2,0.738759,0.754696,0.789414,0.725669
split-3,0.725118,0.735782,0.712678,0.633886
mean,0.756167,0.771778,0.743664,0.690179
std,0.034714,0.038318,0.033021,0.040254


In [17]:
precision

,random forest,XGBoost,CNN,tiny-CNN
Dig,0.773 (0.029),0.777 (0.041),0.812 (0.026),0.748 (0.018)
Eat,0.783 (0.044),0.854 (0.013),0.79 (0.028),0.765 (0.03)
Forward Loco,0.639 (0.128),0.668 (0.163),0.641 (0.093),0.61 (0.111)
Rest,0.916 (0.06),0.874 (0.101),0.766 (0.149),0.751 (0.185)
Stand,0.591 (0.104),0.6 (0.09),0.499 (0.111),0.437 (0.104)
Sweep,0.841 (0.041),0.85 (0.065),0.905 (0.057),0.872 (0.087)
macro avg,0.757 (0.037),0.77 (0.039),0.736 (0.032),0.697 (0.018)
weighted avg,0.764 (0.034),0.788 (0.034),0.757 (0.033),0.713 (0.021)


In [18]:
recall

,random forest,XGBoost,CNN,tiny-CNN
Dig,0.88 (0.003),0.891 (0.025),0.85 (0.009),0.875 (0.024)
Eat,0.743 (0.082),0.726 (0.097),0.695 (0.11),0.604 (0.047)
Forward Loco,0.52 (0.05),0.57 (0.012),0.658 (0.124),0.608 (0.015)
Rest,0.905 (0.104),0.921 (0.084),0.827 (0.044),0.747 (0.143)
Stand,0.652 (0.033),0.72 (0.013),0.581 (0.06),0.497 (0.066)
Sweep,0.692 (0.089),0.717 (0.096),0.818 (0.155),0.762 (0.139)
macro avg,0.732 (0.031),0.757 (0.029),0.738 (0.043),0.682 (0.041)
weighted avg,0.756 (0.043),0.772 (0.047),0.744 (0.04),0.69 (0.049)


In [19]:
f1

,random forest,XGBoost,CNN,tiny-CNN
Dig,0.823 (0.015),0.83 (0.027),0.83 (0.013),0.807 (0.017)
Eat,0.762 (0.059),0.783 (0.061),0.737 (0.076),0.674 (0.025)
Forward Loco,0.572 (0.076),0.608 (0.07),0.642 (0.062),0.605 (0.054)
Rest,0.906 (0.028),0.891 (0.024),0.791 (0.08),0.745 (0.148)
Stand,0.617 (0.066),0.652 (0.057),0.536 (0.089),0.457 (0.055)
Sweep,0.755 (0.038),0.772 (0.036),0.851 (0.067),0.803 (0.054)
macro avg,0.739 (0.034),0.756 (0.036),0.731 (0.036),0.682 (0.036)
weighted avg,0.756 (0.04),0.773 (0.045),0.746 (0.039),0.693 (0.041)
